# Zipline Reloaded Orchestrator Tutorial

This notebook shows how to use `quant-orchestrator` with `quant-warehouse` for multi-vendor, single-symbol and multi-symbol research, Monte Carlo simulation, walk-forward optimization, equity-curve analysis, and portfolio optimization.


In [1]:

from __future__ import annotations

from pathlib import Path
import sys
from itertools import product
from time import perf_counter

import numpy as np
import pandas as pd
from scipy.optimize import minimize

repo_root = Path.cwd().resolve()
if not (repo_root / 'quant_orchestrator').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from quant_orchestrator.experiments import WindowSpec
from quant_orchestrator.monte_carlo import simulate_return_paths
from quant_orchestrator.platforms.backtesting_frameworks.backtesting_py.data_adapter import build_backtesting_frame
from quant_orchestrator.platforms.backtesting_frameworks.shared import MAG7_SYMBOLS, load_price_frame, load_signal_frame, normalize_session_label
from quant_orchestrator.strategy import summarize_backtest, summarize_equity
from quant_warehouse.feature_engineering import compute_features_worldclass

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 20)

SINGLE_SYMBOL = 'AAPL'
MULTI_VENDOR_PROVIDERS = ('fmp', 'yfinance')
TRAIN_START = '2020-01-01'
TRAIN_END = '2025-12-31'
TEST_START = '2026-01-01'
TEST_END = None
FAST_WINDOWS = (10, 20, 50)
SLOW_WINDOWS = (100, 150, 200)


def build_sma_frame(prices: pd.DataFrame, *, fast_window: int, slow_window: int) -> pd.DataFrame:
    if fast_window >= slow_window:
        raise ValueError('fast_window must be less than slow_window')

    frame = compute_features_worldclass(prices.copy())
    fast_col = f'SMA{fast_window}'
    slow_col = f'SMA{slow_window}'
    missing = [column for column in (fast_col, slow_col) if column not in frame.columns]
    if missing:
        raise ValueError(
            'Quant Warehouse feature output is missing required SMA columns: '
            f'{missing}. Update quant-warehouse feature engineering first.'
        )

    frame['fast_sma'] = frame[fast_col]
    frame['slow_sma'] = frame[slow_col]
    frame['signal'] = (frame['fast_sma'] > frame['slow_sma']).astype(int).fillna(0)
    frame = frame.dropna(subset=['open', 'high', 'low', 'close', 'volume']).copy()
    return frame



def combine_equity_sleeves(sleeves: list[pd.Series]) -> pd.Series:
    combined_index = pd.Index([])
    for sleeve in sleeves:
        combined_index = combined_index.union(sleeve.index)
    combined = pd.Series(0.0, index=combined_index, name='portfolio_value')
    for sleeve in sleeves:
        reindexed = sleeve.reindex(combined_index).ffill().fillna(sleeve.iloc[0])
        combined = combined.add(reindexed, fill_value=0.0)
    return combined.sort_index()


def align_return_frame(equity_map: dict[str, pd.Series]) -> pd.DataFrame:
    return pd.concat(
        {name: equity.pct_change().fillna(0.0) for name, equity in equity_map.items()},
        axis=1,
    ).fillna(0.0)


def max_sharpe_weights(return_frame: pd.DataFrame) -> pd.Series:
    if return_frame.empty:
        raise ValueError('return_frame cannot be empty')
    columns = list(return_frame.columns)
    mean = return_frame.mean().to_numpy(dtype=float)
    cov = return_frame.cov().to_numpy(dtype=float)
    if len(columns) == 1:
        return pd.Series([1.0], index=columns)

    def neg_sharpe(weights: np.ndarray) -> float:
        portfolio_mean = float(weights @ mean)
        portfolio_vol = float(np.sqrt(weights @ cov @ weights))
        if portfolio_vol <= 0:
            return 1e9
        return -(portfolio_mean / portfolio_vol)

    bounds = [(0.0, 1.0)] * len(columns)
    constraints = ({'type': 'eq', 'fun': lambda weights: float(np.sum(weights)) - 1.0},)
    guess = np.repeat(1.0 / len(columns), len(columns))
    result = minimize(neg_sharpe, guess, bounds=bounds, constraints=constraints, method='SLSQP')
    weights = pd.Series(result.x if result.success else guess, index=columns)
    weights = weights.clip(lower=0.0)
    return weights / weights.sum()


def weighted_equity_from_returns(return_frame: pd.DataFrame, weights: pd.Series, *, capital_base: float) -> pd.Series:
    aligned = return_frame.loc[:, weights.index].fillna(0.0)
    portfolio_returns = aligned.mul(weights, axis=1).sum(axis=1)
    equity = capital_base * (1.0 + portfolio_returns).cumprod()
    return equity.rename('portfolio_value')


def wfo_windows() -> WindowSpec:
    return WindowSpec(
        mode='fixed',
        train_start=TRAIN_START,
        train_end=TRAIN_END,
        test_start=TEST_START,
        test_end=TEST_END,
    )


def run_monte_carlo(equity: pd.Series, *, iterations: int = 1000, block_size: int = 5) -> pd.DataFrame:
    returns = equity.pct_change().dropna()
    mc = simulate_return_paths(returns, iterations=iterations, block_size=block_size, horizon=len(returns))
    return mc.summary


def summarize_candidates(rows: list[dict[str, object]]) -> pd.DataFrame:
    table = pd.DataFrame(rows)
    if table.empty:
        return table
    return table.sort_values(by=['total_return', 'max_drawdown', 'final_equity'], ascending=[False, True, False]).reset_index(drop=True)

FRAMEWORK_TITLE = 'Zipline Reloaded'


def _patch_zipline_compatibility() -> None:
    if getattr(_patch_zipline_compatibility, '_patched', False):
        return

    import zipline.finance.ledger as ledger_mod
    from zipline.data.in_memory_daily_bars import InMemoryDailyBarReader

    InMemoryDailyBarReader.frames = property(lambda self: self._frames)
    ledger_mod.PositionTracker.stats = property(lambda self: self._stats)
    _patch_zipline_compatibility._patched = True  # type: ignore[attr-defined]


def run_framework_backtest(frame: pd.DataFrame, *, symbol: str, capital_base: float):
    from zipline.algorithm import TradingAlgorithm
    from zipline.api import order_target, record, symbol as zipline_symbol
    from quant_orchestrator.platforms.backtesting_frameworks.zipline.data_adapter import build_zipline_in_memory_data

    _patch_zipline_compatibility()
    adapter = build_zipline_in_memory_data(frame, symbol=symbol, capital_base=capital_base)
    trade_size = adapter.trade_size

    def initialize(context, **kwargs):
        context.asset = zipline_symbol(symbol.upper())
        context.is_long = False

    def handle_data(context, data):
        dt = normalize_session_label(context.get_datetime())
        bullish = adapter.signal_map.get(dt, False)
        if bullish and not context.is_long:
            order_target(context.asset, trade_size)
            context.is_long = True
        elif not bullish and context.is_long:
            order_target(context.asset, 0)
            context.is_long = False
        record(price=data.current(context.asset, 'price'), signal=float(bullish))

    started = perf_counter()
    perf = TradingAlgorithm(
        sim_params=adapter.sim_params,
        data_portal=adapter.data_portal,
        asset_finder=adapter.asset_finder,
        initialize=initialize,
        handle_data=handle_data,
        capital_base=capital_base,
        benchmark_returns=adapter.benchmark_returns,
    ).run()
    elapsed = perf_counter() - started
    equity = perf['portfolio_value'].rename('portfolio_value')
    transactions = perf.get('transactions', pd.Series(index=perf.index, data=[[]] * len(perf)))
    summary = summarize_backtest(
        framework='zipline',
        symbol=symbol,
        equity=equity,
        elapsed_seconds=elapsed,
        bars=len(perf),
        trades=int(transactions.map(len).sum()),
    )
    summary['native_last_value'] = float(perf['portfolio_value'].iloc[-1])
    return perf, summary, equity


## Single Symbol / Multi Vendor


In [2]:

print(f'Tutorial framework: {FRAMEWORK_TITLE}')
print(f'Single symbol: {SINGLE_SYMBOL}')
print(f'Vendors: {MULTI_VENDOR_PROVIDERS}')


Tutorial framework: Zipline Reloaded
Single symbol: AAPL
Vendors: ('fmp', 'yfinance')


## Multi Vendor Backtest


In [3]:

def run_multi_vendor_single_symbol(symbol: str = SINGLE_SYMBOL, providers: tuple[str, ...] = MULTI_VENDOR_PROVIDERS, capital_base: float = 100_000.0) -> pd.DataFrame:
    rows = []
    for provider in providers:
        frame = load_signal_frame(symbol, provider=provider, start=TRAIN_START, end=TEST_END)
        _, summary, _ = run_framework_backtest(frame, symbol=symbol, capital_base=capital_base)
        rows.append(summary.assign(provider=provider))
    return pd.concat(rows, ignore_index=True)

vendor_report = run_multi_vendor_single_symbol()
display(vendor_report.round(4).head(10))


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


,framework,symbol,start,end,bars,trades,final_equity,total_return,max_drawdown,daily_vol,elapsed_seconds,bars_per_second,native_last_value,provider
0,zipline,AAPL,2020-10-15,2026-06-24,1428,10,109963.02,0.0996,-0.5538,0.0485,0.2917,4895.56,109963.0186,fmp
1,zipline,AAPL,2020-10-15,2026-06-23,1427,10,111417.66,0.1142,-0.5331,0.0456,0.2538,5621.92,111417.6626,yfinance


## Multi Symbol Backtest


In [4]:

def run_multi_symbol_backtest(symbols: tuple[str, ...] = MAG7_SYMBOLS, provider: str = 'fmp', capital_base: float = 100_000.0):
    capital_per_symbol = capital_base / max(1, len(symbols))
    symbol_rows = []
    sleeves = []
    runs = {}
    elapsed = 0.0
    trades = 0
    for symbol in symbols:
        frame = load_signal_frame(symbol, provider=provider, start=TRAIN_START, end=TEST_END)
        started = perf_counter()
        raw, summary, equity = run_framework_backtest(frame, symbol=symbol, capital_base=capital_per_symbol)
        elapsed += perf_counter() - started
        symbol_rows.append(summary.assign(provider=provider))
        sleeves.append(equity.rename(symbol))
        trades += int(summary['trades'].iloc[0])
        runs[symbol] = raw
    portfolio_equity = combine_equity_sleeves(sleeves)
    portfolio_summary = summarize_backtest(
        framework=FRAMEWORK_TITLE,
        symbol='MAG7',
        equity=portfolio_equity,
        elapsed_seconds=elapsed,
        bars=len(portfolio_equity),
        trades=trades,
    )
    portfolio_summary['provider'] = provider
    return pd.concat(symbol_rows, ignore_index=True), portfolio_summary, runs, portfolio_equity

symbol_report, portfolio_report, runs_by_symbol, portfolio_equity = run_multi_symbol_backtest()
print('Symbol report:')
display(symbol_report.loc[:, ['provider', 'symbol', 'start', 'end', 'bars', 'trades', 'final_equity', 'total_return', 'max_drawdown', 'elapsed_seconds']].round(4).head(10))
print()
print('Portfolio report:')
display(portfolio_report.round(4).head(10))


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


...

/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


Symbol report:


,provider,symbol,start,end,bars,trades,final_equity,total_return,max_drawdown,elapsed_seconds
0,fmp,AAPL,2020-10-15,2026-06-24,1428,10,15688.96,0.0982,-0.5464,0.3212
1,fmp,AMZN,2020-10-15,2026-06-24,1428,10,13798.24,-0.0341,-0.4023,0.2566
2,fmp,GOOGL,2020-10-15,2026-06-24,1428,6,25978.12,0.8185,-0.4545,0.2545
3,fmp,META,2020-10-15,2026-06-24,1428,6,19930.68,0.3951,-0.4420,0.2523
4,fmp,MSFT,2020-10-15,2026-06-24,1428,10,16473.52,0.1531,-0.4757,0.3197
5,fmp,NVDA,2020-10-15,2026-06-24,1428,6,51411.29,2.5988,-0.9837,0.2572
6,fmp,TSLA,2020-10-15,2026-06-24,1428,12,14149.40,-0.0095,-0.6251,0.2558



Portfolio report:


,framework,symbol,start,end,bars,trades,final_equity,total_return,max_drawdown,daily_vol,elapsed_seconds,bars_per_second,provider
0,Zipline Reloaded,MAG7,2020-10-15,2026-06-24,1428,60,157430.2,0.5743,-0.613,0.0336,2.0459,697.98,fmp


## Monte Carlo


In [5]:

mc_report = run_monte_carlo(portfolio_equity, iterations=1000, block_size=5)
display(mc_report.round(4).head(10))


,iterations,horizon,terminal_return_mean,terminal_return_p05,terminal_return_p50,terminal_return_p95,max_drawdown_mean,max_drawdown_p05
0,1000,1427,1.597,-0.6803,0.5124,6.2808,-0.5329,-0.8109


## Walk Forward Optimization


In [6]:

def run_walk_forward_optimization(symbol: str = SINGLE_SYMBOL, provider: str = 'fmp', capital_base: float = 100_000.0):
    spec = wfo_windows()
    train_frame = load_price_frame(symbol, provider=provider, start=spec.train_start, end=spec.train_end)
    test_frame = load_price_frame(symbol, provider=provider, start=spec.test_start, end=spec.test_end)

    train_rows = []
    for fast_window, slow_window in product(FAST_WINDOWS, SLOW_WINDOWS):
        if fast_window >= slow_window:
            continue
        frame = build_sma_frame(train_frame, fast_window=fast_window, slow_window=slow_window)
        _, summary, _ = run_framework_backtest(frame, symbol=symbol, capital_base=capital_base)
        row = summary.iloc[0].to_dict()
        row['fast_window'] = fast_window
        row['slow_window'] = slow_window
        train_rows.append(row)

    train_grid = summarize_candidates(train_rows)
    best = train_grid.iloc[0]
    best_fast = int(best['fast_window'])
    best_slow = int(best['slow_window'])
    _, _, test_equity = run_framework_backtest(build_sma_frame(test_frame, fast_window=best_fast, slow_window=best_slow), symbol=symbol, capital_base=capital_base)
    test_summary = summarize_backtest(
        framework=FRAMEWORK_TITLE,
        symbol=symbol,
        equity=test_equity,
        elapsed_seconds=0.0,
        bars=len(test_frame),
        trades=None,
    )
    return train_grid, best, test_summary

train_grid, best_row, test_summary = run_walk_forward_optimization()
print('Train grid:')
display(train_grid.loc[:, ['fast_window', 'slow_window', 'final_equity', 'total_return', 'max_drawdown']].round(4).head(10))
print()
print('Best parameters:')
display(best_row.loc[['fast_window', 'slow_window', 'total_return', 'max_drawdown']].to_frame(name='value').round(4))
print()
print('Test summary:')
display(test_summary.round(4))


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


...

,fast_window,slow_window,final_equity,total_return,max_drawdown
0,10,200,141797.01,0.4180,-0.6226
1,50,100,141778.08,0.4178,-0.6421
2,20,100,141783.93,0.4178,-0.6385
3,20,200,135079.85,0.3508,-0.6556
4,10,100,132303.50,0.3230,-0.6983
5,10,150,130058.18,0.3006,-0.7081
6,50,150,125611.92,0.2561,-0.7223
7,20,150,125262.15,0.2526,-0.7158
8,50,200,108640.43,0.0864,-0.8641



Best parameters:


,value
fast_window,10
slow_window,200
total_return,0.418
max_drawdown,-0.6226



Test summary:


,framework,symbol,start,end,bars,trades,final_equity,total_return,max_drawdown,daily_vol,elapsed_seconds,bars_per_second
0,Zipline Reloaded,AAPL,2026-01-02,2026-06-24,119,None,100000.0,0.0,0.0,0.0,0.0,None


## Equity Curve Analysis


In [7]:

summary_view = pd.Series(summarize_equity(portfolio_equity), name='value').to_frame()
display(summary_view.round(4))
display(portfolio_equity.tail().to_frame(name='portfolio_value').round(2))


,value
start,2020-10-15
end,2026-06-24
final_equity,157430.2
total_return,0.5743
max_drawdown,-0.613
daily_vol,0.0336


,portfolio_value
2026-06-17 20:00:00+00:00,77435.08
2026-06-18 20:00:00+00:00,77435.08
2026-06-22 20:00:00+00:00,77435.08
2026-06-23 20:00:00+00:00,77435.08
2026-06-24 20:00:00+00:00,157430.20


## Portfolio Optimization


In [8]:

def optimize_symbol_portfolio(symbols: tuple[str, ...] = MAG7_SYMBOLS, provider: str = 'fmp', capital_base: float = 100_000.0):
    capital_per_symbol = capital_base / max(1, len(symbols))
    equity_map = {}
    for symbol in symbols:
        frame = load_signal_frame(symbol, provider=provider, start=TRAIN_START, end=TEST_END)
        _, _, equity = run_framework_backtest(frame, symbol=symbol, capital_base=capital_per_symbol)
        equity_map[symbol] = equity
    return equity_map

symbol_equities = optimize_symbol_portfolio()
symbol_returns = align_return_frame(symbol_equities)
symbol_weights = max_sharpe_weights(symbol_returns)
symbol_portfolio_equity = weighted_equity_from_returns(symbol_returns, symbol_weights, capital_base=100_000.0)
print('Symbol-level weights:')
display(symbol_weights.sort_values(ascending=False).to_frame(name='weight').round(4))
print()
print('Symbol-level portfolio summary:')
display(pd.Series(summarize_equity(symbol_portfolio_equity), name='value').to_frame().round(4))

candidate_pairs = [(fast, slow) for fast in FAST_WINDOWS for slow in SLOW_WINDOWS if fast < slow]
strategy_equities = {}
strategy_rows = []
strategy_frame = load_price_frame(SINGLE_SYMBOL, provider='fmp', start=TRAIN_START, end=TEST_END)
for fast_window, slow_window in candidate_pairs:
    candidate_frame = build_sma_frame(strategy_frame, fast_window=fast_window, slow_window=slow_window)
    _, summary, equity = run_framework_backtest(candidate_frame, symbol=SINGLE_SYMBOL, capital_base=100_000.0)
    key = f'{fast_window}/{slow_window}'
    strategy_equities[key] = equity
    strategy_rows.append({
        'strategy': key,
        'fast_window': fast_window,
        'slow_window': slow_window,
        'total_return': float(summary['total_return'].iloc[0]),
        'max_drawdown': float(summary['max_drawdown'].iloc[0]),
    })
strategy_returns = align_return_frame(strategy_equities)
strategy_weights = max_sharpe_weights(strategy_returns)
strategy_portfolio_equity = weighted_equity_from_returns(strategy_returns, strategy_weights, capital_base=100_000.0)
print()
print('Strategy-level weights:')
display(strategy_weights.sort_values(ascending=False).head(10).to_frame(name='weight').round(4))
print()
print('Strategy-level portfolio summary:')
display(pd.Series(summarize_equity(strategy_portfolio_equity), name='value').to_frame().round(4))


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


...

/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(



Strategy-level weights:


,weight
20/100,0.3092
10/100,0.2956
10/150,0.2410
20/150,0.0827
50/100,0.0714
20/200,0.0000
10/200,0.0000
50/150,0.0000
50/200,0.0000



Strategy-level portfolio summary:


,value
start,2020-01-02
end,2026-06-24
final_equity,61515154.23
total_return,614.1515
max_drawdown,-0.5725
daily_vol,0.1039
